In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList


class BanWordsLogitsProcessor(LogitsProcessor):

    def __init__(self, tokenizer, target_word="UNIX", max_occurrences=1, top_k=50):
        self.max_occurrences = max_occurrences
        self.top_k = top_k

        self.seq = tokenizer.encode(" " + target_word, add_special_tokens=False)
        self.seq_no_space = tokenizer.encode(target_word, add_special_tokens=False)

    def _count_full(self, ids):
        count = 0
        for seq in (self.seq, self.seq_no_space):
            n = len(seq)
            for i in range(len(ids) - n + 1):
                if ids[i:i + n] == seq:
                    count += 1
        return count

    def _prefix_len(self, ids):
        best = 0
        for seq in (self.seq, self.seq_no_space):
            for k in range(min(len(seq) - 1, len(ids)), 0, -1):
                if ids[-k:] == seq[:k]:
                    return max(best, k)
        return best

    def __call__(self, input_ids, scores):
        ids = input_ids[0].tolist()

        if self._count_full(ids) >= self.max_occurrences:
            return scores

        prefix_len = self._prefix_len(ids)

        if prefix_len == 0:
            firsts = {self.seq[0], self.seq_no_space[0]}
            top_ids = torch.topk(scores[0], self.top_k).indices.tolist()
            to_ban = [t for t in top_ids if t not in firsts]
            scores[:, to_ban] = -float("inf")
            return scores

        allowed = set()
        for seq in (self.seq, self.seq_no_space):
            if prefix_len < len(seq) and ids[-prefix_len:] == seq[:prefix_len]:
                allowed.add(seq[prefix_len])
        if allowed:
            mask = torch.full_like(scores, -float("inf"))
            mask[:, list(allowed)] = scores[:, list(allowed)]
            return mask

        return scores

In [2]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [14]:
import re
processor = BanWordsLogitsProcessor(tokenizer, "Skyrim", max_occurrences=1, top_k=37)
logits_processor = LogitsProcessorList([processor])

inputs = tokenizer("The most popular game ever is", return_tensors="pt")
outputs = model.generate(
    **inputs, max_new_tokens=60, do_sample=False,
    logits_processor=logits_processor, pad_token_id=tokenizer.eos_token_id,
)
text = tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
print(text)
match_exact = re.search(r'\bSkyrim\b', text) is not None

print("Skyrim:", match_exact)

The most popular game ever is Skyrim. It's a game that's been around for a long time, and it's been a huge hit. It's a game that's been around for a long time, and it's been a huge hit. It's a game that's been around for a long time, and it's
Skyrim: True
